# speech_llm : toutes les étapes sur Colab (GPU T4)
Avant tout : *Exécution → Modifier le type d'exécution → GPU T4*, puis glisse `speech_proj.zip` dans `/content`.

| Section | Étape | État du code |
|---|---|---|
| 1 | A : environnement | prêt |
| 2 | B : tests avec faux modèles | prêt |
| 3 | Données dev | prêt |
| 4 | C : traitement audio (vrai extracteur) | prêt |
| 5 | D : longueurs des cibles (vrai tokenizer) | prêt |
| 6 | Drive | prêt |
| 7 | F : essai de 50 pas | prêt |
| 8 | Stade 1 : entraînement du projecteur | prêt |
| 9 | Courbes | prêt |
| 10 | Références françaises (NLLB) | prêt, jamais lancé |
| 11 | Stade 2 : projecteur + LoRA | prêt, jamais lancé |
| 12 | E : projecteur causal | à écrire |
| 13 | G : streaming | à écrire |

Si la session est coupée : relance les sections 1, 3 et 6, puis reprends où tu en étais.

## 1. Étape A : environnement
`torchao` est retiré : la version de Colab est trop ancienne pour `peft`, et on ne s'en sert pas.

In [ ]:
!nvidia-smi -L
import os, sys, torch
print(sys.version.split()[0], torch.__version__, 'cuda', torch.cuda.is_available())
assert os.path.exists('/content/speech_proj.zip'), 'glisse speech_proj.zip dans /content'
!cd /content && unzip -q -o speech_proj.zip
!pip uninstall -y -q torchao
!pip install -q peft librosa soundfile sacrebleu sentencepiece accelerate pytest
%cd /content/speech_proj
!ls train.py make_french.py manifest_dev_train.jsonl

## 2. Étape B : tests (faux modèles, quelques secondes)

In [ ]:
!python -m pytest -q -p no:cacheprovider 2>&1 | tail -15

## 3. Données dev (1 Go)
Ajoute `HF_TOKEN` (jeton en lecture) dans les Secrets de Colab et autorise l'accès au notebook.

In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import tarfile
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
p = hf_hub_download('QCRI/mgb2', 'dev/dev.tar.gz', repo_type='dataset')
os.makedirs('/content/dev/wav', exist_ok=True)
with tarfile.open(p) as t:
    for m in t:
        if m.isfile() and m.name.endswith('.wav'):
            m.name = os.path.basename(m.name)
            t.extract(m, '/content/dev/wav')
print(len(os.listdir('/content/dev/wav')), 'fichiers wav')

## 4. Étape C : traitement audio avec le vrai extracteur Whisper
Affiche, pour 5 segments : durée, forme des features, frames utiles et nombre de tokens audio.

In [ ]:
import json
from transformers import WhisperFeatureExtractor
from speech_llm.audio import load_segment, extract_features, useful_encoder_frames
from speech_llm.projector import audio_token_count
fe = WhisperFeatureExtractor.from_pretrained('openai/whisper-large-v3')
rows = [json.loads(l) for l in open('manifest_dev_val.jsonl', encoding='utf-8')][:5]
wavs = [load_segment(r) for r in rows]
feats, mel = extract_features(fe, wavs)
print('features', tuple(feats.shape), '(attendu : 5, 128, 3000)')
for r, w, m in zip(rows, wavs, mel):
    enc = useful_encoder_frames(int(m))
    print(f'{len(w)/16000:5.2f} s | mel utiles {int(m):4d} | frames encodeur {enc:4d} | tokens audio {audio_token_count(enc):3d}')

## 5. Étape D : longueurs des cibles arabes avec le vrai tokenizer Qwen

In [ ]:
import numpy as np
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')
for name in ['manifest_dev_train.jsonl', 'manifest_dev_val.jsonl']:
    n = [len(tok(json.loads(l)['text_ar'], add_special_tokens=False).input_ids) for l in open(name, encoding='utf-8')]
    print(name, '| moyenne %.1f | p99 %d | max %d | > 96 tokens : %d' % (np.mean(n), np.percentile(n, 99), max(n), sum(x > 96 for x in n)))

## 6. Sauvegarde sur Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/speech_llm/stage1 /content/drive/MyDrive/speech_llm/stage2

## 7. Étape F : essai court de 50 pas
Regarde que la perte baisse. En cas de `CUDA out of memory`, passe `--batch 4`.

In [ ]:
!python train.py --stage 1 --train-manifest manifest_dev_train.jsonl --val-manifest manifest_dev_val.jsonl --max-steps 50 --eval-every 25 --batch 8 --out /content/ckpt_essai

## 8. Stade 1 : entraînement du projecteur (transcription arabe, Qwen gelé)
Reprise après coupure : ajoute `--resume /content/drive/MyDrive/speech_llm/stage1/last.pt`.

In [ ]:
!python train.py --stage 1 --train-manifest manifest_dev_train.jsonl --val-manifest manifest_dev_val.jsonl --epochs 3 --batch 8 --eval-every 100 --out /content/drive/MyDrive/speech_llm/stage1

## 9. Courbes (relance avec `d` = stage2 après le stade 2)

In [ ]:
import json, matplotlib.pyplot as plt
d = '/content/drive/MyDrive/speech_llm/stage1'
tr = [json.loads(l) for l in open(d + '/log.jsonl')]
va = [json.loads(l) for l in open(d + '/val.jsonl')]
plt.plot([r['step'] for r in tr], [r['loss'] for r in tr], alpha=.4, label='entraînement')
plt.plot([r['step'] for r in va], [r['val_loss'] for r in va], 'o-', label='validation')
plt.xlabel('étape'); plt.ylabel('perte'); plt.legend(); plt.show()
print(va[-1]['val_wer_sample'], va[-1]['samples'])

## 10. Références françaises (NLLB-200 600M, GPU)
Traduit chaque segment arabe en français. Ce sont des **traductions automatiques approximatives** (segment par segment, sans contexte), pas de vraies références. Essai sur 64 segments d'abord, puis tout.

In [ ]:
!python make_french.py manifest_dev_val.jsonl /content/essai_fr.jsonl --limit 64
rows = [json.loads(l) for l in open('/content/essai_fr.jsonl', encoding='utf-8')]
for r in rows[:5]:
    print(r['text_ar'], '\n  ->', r['text_fr'])

In [ ]:
!python make_french.py manifest_dev_train.jsonl /content/drive/MyDrive/speech_llm/manifest_dev_train_fr.jsonl
!python make_french.py manifest_dev_val.jsonl /content/drive/MyDrive/speech_llm/manifest_dev_val_fr.jsonl

## 11. Stade 2 : projecteur + LoRA, transcription et traduction vers le français
Part du projecteur du stade 1 (`--init`, poids seulement). Nécessite les sections 6, 8 et 10.

In [ ]:
!python train.py --stage 2 --tasks asr,st --init /content/drive/MyDrive/speech_llm/stage1/last.pt \
  --train-manifest /content/drive/MyDrive/speech_llm/manifest_dev_train_fr.jsonl \
  --val-manifest /content/drive/MyDrive/speech_llm/manifest_dev_val_fr.jsonl \
  --epochs 3 --batch 4 --eval-every 100 --out /content/drive/MyDrive/speech_llm/stage2

## 12. Étape E : projecteur causal (inférence par blocs)
Pas encore écrit. Cette section sera remplie quand le code et ses tests existeront.

## 13. Étape G : streaming
Pas encore écrit. Dépend de l'étape E.